# Website Trafik Tahmini

Blog sitesinin günlük görüntülenmesini tahmin edeceğim.


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns


### Data


In [ ]:
df=pd.read_csv('data/Thecleverprogrammer.csv')
df.head()


### EDA


In [ ]:
df.info()
df.isnull().sum()


### Görselleştirme


In [ ]:
df['Date']=pd.to_datetime(df['Date'],dayfirst=True)
plt.plot(df['Date'],df['Views'])
plt.show()


### Boş veri


In [ ]:
df['Views']=df['Views'].ffill()


### Feature Engineering


In [ ]:
s=df.sort_values('Date').copy()
s['lag1']=s['Views'].shift(1)
s['lag7']=s['Views'].shift(7)
s['dow']=s['Date'].dt.dayofweek
s=s.dropna()


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
x=s[['lag1','lag7','dow']]
y=s['Views']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,shuffle=False)


### Modelling


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.metrics import r2_score,mean_absolute_error


In [ ]:
lr=LinearRegression()
lr.fit(x_train,y_train)
print('LR',round(r2_score(y_test,lr.predict(x_test)),3),round(mean_absolute_error(y_test,lr.predict(x_test)),1))


In [ ]:
rf=RandomForestRegressor(random_state=42)
rf.fit(x_train,y_train)
print('RF',round(r2_score(y_test,rf.predict(x_test)),3),round(mean_absolute_error(y_test,rf.predict(x_test)),1))


In [ ]:
gb=GradientBoostingRegressor(random_state=42)
gb.fit(x_train,y_train)
print('GBM',round(r2_score(y_test,gb.predict(x_test)),3),round(mean_absolute_error(y_test,gb.predict(x_test)),1))


### Feature Importance + Residual


In [ ]:
rf=RandomForestRegressor(random_state=42).fit(x_train,y_train)
print(pd.Series(rf.feature_importances_,index=x.columns))
pred=rf.predict(x_test)
plt.scatter(pred,y_test-pred)
plt.axhline(0,color='r')
plt.show()


In [ ]:
import joblib
joblib.dump(rf,'../../models/timeseries_website_traffic.joblib')


### Sonuç

lag7 işe yaradı, haftalık ritim var. GBM biraz daha iyi ama RF de yakın.
